## Data Availability

This tutorial uses the **Human Immune** dataset.
Download from Google Drive or use the built-in mock data fallback.

```bash
# Download from: https://drive.google.com/drive/folders/11PuJRaw_JC7xJhphQklac7eg9AuGHsCE
```


# Tutorial: Batch integration with neighbor graphs

This tutorial shows how to use Cytoneighbor to **integrate
multi-batch single-cell data** and examine how the graph organizes related cell types.

You will learn:

1. How to build a Cytoneighbor neighbour graph across batches
2. How to **visualize** integration quality
3. How to use the graph for **downstream analysis**
   (clustering, marker detection, UMAP)
4. How to **compare** neighbor graphs
5. How to **adapt** this workflow to your own multi-batch data


## 1. Setup


In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "pyproject.toml").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if (PROJECT_ROOT / "cytoneighbor").is_dir():
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import warnings
import cytoneighbor as cn

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, facecolor="white")


## 2. Load multi-batch data

The cell below loads real data or generates a synthetic dataset
with multiple batches for demonstration.


In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "Human immune.h5ad"
USE_MOCK = False

if DATA_PATH.is_file():
    adata = sc.read_h5ad(DATA_PATH)
    print(f"Loaded real dataset: {adata.n_obs} cells x {adata.n_vars} genes")
else:
    USE_MOCK = True
    warnings.warn(
        f"Dataset not found. Generating synthetic mock data. "
        "Download the real dataset for production use."
    )
    
    rng = np.random.default_rng(42)
    N_CELLS, N_GENES, N_BATCHES = 300, 600, 5
    
    batches = np.array([f"Batch_{i % N_BATCHES}" for i in range(N_CELLS)])
    
    type_list = ["CD4+ T cells", "CD8+ T cells", "NK cells", "NKT cells",
                 "B cells", "Monocytes", "Dendritic cells"]
    cell_types = np.array([type_list[i % 7] for i in range(N_CELLS)])
    
    type_ids = np.array([type_list.index(ct) for ct in cell_types])
    type_means = rng.normal(0, 1, size=(len(type_list), N_GENES))
    batch_list = sorted(set(batches))
    batch_ids = np.array([batch_list.index(b) for b in batches])
    batch_effects = rng.normal(0, 0.3, size=(len(batch_list), N_GENES))
    cell_noise = rng.normal(0, 0.5, size=(N_CELLS, N_GENES))
    X = type_means[type_ids] + batch_effects[batch_ids] + cell_noise
    X -= X.min(axis=0, keepdims=True); X += 0.1
    
    gene_names = [f"Gene_{j}" for j in range(N_GENES)]
    adata = sc.AnnData(X.astype(np.float32),
                       obs={"batch": batches, "final_annotation": cell_types},
                       dtype=np.float32)
    adata.var_names = gene_names
    print(f"Generated mock dataset: {adata.n_obs} cells x {adata.n_vars} genes")

print(f"Batches: {adata.obs['batch'].nunique()}  |  "
      f"Cell types: {adata.obs['final_annotation'].nunique()}")
print(f"Cells per batch: {adata.obs['batch'].value_counts().to_dict()}")


## 3. Build a Cytoneighbor neighbour graph

Cytoneighbor constructs a **pairwise-score-based neighbour graph**
from pairwise scores. Use it in Scanpy to examine batch mixing
and cell type structure as separate outcomes.


In [ ]:
# Standard preprocessing
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=3000, batch_key="batch")

print(f"Highly variable genes selected: {adata.var.highly_variable.sum()}")


In [ ]:
# Train Cytoneighbor on the full dataset
X_data = adata.X
if hasattr(X_data, "toarray"):
    X_data = X_data.toarray()
labels = adata.obs["final_annotation"].values
cn.load_data(adata, data=X_data, labels=labels)
if USE_MOCK:
    cn.fit(adata, epochs=2, d_model=64, depth=1, hidden=128,
           train_pairs_per_epoch=200, use_cuda=False, use_amp=False,
           synth_enable=False, validation_split=None, early_stopping_patience=None)
else:
    cn.fit(adata, epochs=1, early_stopping_patience=5)

# Build the neighbour graph
cn.transform(adata, n_neighbors=15)
print("Neighbour graph built: cytoneighbor_connectivities in adata.obsp")

In [ ]:
# Compute UMAP using the Cytoneighbor graph
sc.tl.pca(adata, n_comps=50)
sc.tl.umap(adata, neighbors_key="neighbors", init_pos='random')
print("UMAP computed from Cytoneighbor graph.")

## 4. Visualize integration quality

The key question for any integration method: **does it mix
batches while preserving cell-type separation?**

We compare the UMAP coloured by batch vs. cell type.


In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt

# 全局设置（可选）
sc.settings.set_figure_params(dpi=150, fontsize=10, facecolor='white')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 左图：Batch mixing
sc.pl.umap(
    adata,
    color="batch",
    ax=axes[0],
    show=False,
    size=80,                    
    alpha=0.8,                
    palette="tab10",            
    frameon=True,              
    title="Batch mixing",
    legend_loc="right margin",  
)

# 右图：Cell-type structure
sc.pl.umap(
    adata,
    color="final_annotation",
    ax=axes[1],
    show=False,
    size=100,
    alpha=0.8,
    palette="Set2",            
    frameon=True,
    title="Cell-type structure",
    legend_loc="right margin",
)

plt.tight_layout()
plt.show()

## 5. Downstream analysis with the Cytoneighbor graph

The `cytoneighbor_connectivities` graph is a standard Scanpy
neighbour graph. You can use it for **any** downstream task:
clustering, marker gene detection, pseudotime, etc.


In [ ]:
# Leiden clustering using the Cytoneighbor graph
sc.tl.leiden(adata, neighbors_key="neighbors",
             resolution=0.8)

sc.pl.umap(adata, color="leiden", size=70, title="Leiden clusters (Cytoneighbor graph)",
           legend_loc="right margin")

# Find marker genes per cluster
sc.tl.rank_genes_groups(adata, groupby="leiden", method="wilcoxon")
print("Top markers for cluster 0:")
print(pd.DataFrame(adata.uns["rank_genes_groups"]["names"]).head(5))


## 6. Compare neighbor graphs

For a quantitative assessment, you can compute **iLISI** (batch
mixing) and **cLISI** (local diversity of cell type annotations) scores.
We also compare against a baseline: UMAP from the raw PCA space
without any batch correction.


In [ ]:
# Baseline: UMAP without batch correction
ad_base = adata.copy()
sc.pp.neighbors(ad_base, n_neighbors=15, use_rep="X_pca")
sc.tl.umap(ad_base, neighbors_key="neighbors", init_pos="random")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sc.pl.umap(adata, color="batch", ax=axes[0], show=False, size=70,
           title="Uncorrected (PCA neighbors)", legend_loc="right margin")
sc.pl.umap(adata, color="final_annotation", ax=axes[1], show=False, size=70,
           title="Cell types (uncorrected)", legend_loc="right margin")
plt.tight_layout(); plt.show()

# Descriptive UMAP-based LISI for this tutorial
import harmonypy as hm

def compute_lisi_metrics(adata, batch_key, cell_type_key, embed_key=None, n_components=20):
    """Compute descriptive iLISI and cLISI on the tutorial UMAP."""
    adata = adata.copy()  # avoid mutating the original
    if embed_key and embed_key in adata.obsm:
        latent = adata.obsm[embed_key]
    else:
        sc.tl.umap(adata, n_components=n_components, init_pos='random')
        latent = adata.obsm['X_umap']
    
    meta_df = adata.obs[[batch_key, cell_type_key]].copy()
    res = hm.lisi.compute_lisi(latent, meta_df, label_colnames=[batch_key, cell_type_key])
    return float(np.nanmean(res[:, 0])), float(np.nanmean(res[:, 1]))


# -------------------------------------------------------------------------
# 1. Uncorrected baseline (recomputed on a copy so we don't lose Cytoneighbor)
# -------------------------------------------------------------------------
print("[1/2] Computing Uncorrected baseline metrics...")
ilisi_base, clisi_base = compute_lisi_metrics(ad_base, "batch", "final_annotation", "X_umap")
del ad_base

# -------------------------------------------------------------------------
# 2. Cytoneighbor (current adata already has the Cytoneighbor graph)
# -------------------------------------------------------------------------
print("[2/2] Computing Cytoneighbor metrics...")
ilisi_cy, clisi_cy = compute_lisi_metrics(adata, "batch", "final_annotation", "X_umap")

# -------------------------------------------------------------------------
# 3. Side-by-side comparison
# -------------------------------------------------------------------------
results = pd.DataFrame({
    "Method": ["Uncorrected", "Cytoneighbor"],
    "iLISI (Batch Mixing ↑)": [ilisi_base, ilisi_cy],
    "cLISI (Local Label Diversity ↓)": [clisi_base, clisi_cy],
})

print("\n" + "="*55)
print("Integration Metrics Comparison")
print("="*55)
print(results.to_string(index=False))
print("="*55)

# -------------------------------------------------------------------------
# 4. Auto-interpretation
# -------------------------------------------------------------------------
n_batches = adata.obs["batch"].nunique()
print(f"\nReference values for this dataset:")
print(f"  • n_batches = {n_batches}")
print(f"  • iLISI ideal (perfect mixing) ≈ {n_batches:.1f}")
print(f"  • cLISI ideal (perfect purity)  ≈ 1.0")

print("\nCytoneighbor vs Uncorrected:")
delta_i = ilisi_cy - ilisi_base
delta_c = clisi_cy - clisi_base

if delta_i > 0:
    print(f"  ✓ Batch mixing  (iLISI):  +{delta_i:.4f}  ({ilisi_base:.4f} → {ilisi_cy:.4f})")
else:
    print(f"  ⚠ Batch mixing  (iLISI):  {delta_i:+.4f}  ({ilisi_base:.4f} → {ilisi_cy:.4f})")

if delta_c < 0:
    print(f"  ✓ Label diversity (cLISI):  {delta_c:.4f}  ({clisi_base:.4f} → {clisi_cy:.4f})")
else:
    print(f"  ⚠ Label diversity (cLISI):  +{delta_c:.4f}  ({clisi_base:.4f} → {clisi_cy:.4f})")

print("\nFor full multi-method benchmarks (Harmony, scVI, BBKNN):")
print("  python experiments/Rebatch/a-main_remove.py")

## 7. Adapt to your own data

To integrate your own multi-batch dataset, replace the data
loading step and specify your column names:

```python
import cytoneighbor as cn
import scanpy as sc

# 1. Load YOUR data
adata = sc.read_h5ad("your_multi_batch_data.h5ad")

# 2. Preprocess (standard Scanpy pipeline)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=3000,
                            batch_key="your_batch_column")

# 3. Train Cytoneighbor
cn.load_data(adata, data=adata.X, labels=adata.obs["your_cell_type_column"].to_numpy())
cn.fit(adata)

# 4. Build the neighbour graph
cn.transform(adata, n_neighbors=15)

# 5. Use the graph for ANY downstream Scanpy function
sc.tl.umap(adata, neighbors_key="neighbors")
sc.tl.leiden(adata, neighbors_key="neighbors")
```

The `neighbors_key="neighbors"` parameter
is all you need to plug Cytoneighbor into your existing Scanpy
workflow.


## Summary

You have learned how to:

- Build a **Cytoneighbor neighbour graph** on multi-batch data
- **Visualize** integration quality with batch/cell-type UMAPs
- Use the graph for **downstream analysis** (clustering, markers)
- **Compare** results against a PCA neighbor baseline
- **Adapt** the workflow to your own data

Next steps:
- Try the [Annotation tutorial](tutorial_annotation.ipynb)
  to learn how Cytoneighbor handles cell-type transfer
- See the [reproduction guide](../repro/) for the original experiments and the revision analysis replay
